# TP3 - SVM on heart sounds (Mel spectrogram + 3-fold cross-validation)

This notebook redoes the **professor's TP3** (`TP3.ipynb`, written for Google Colab, not in this repo) on our local machine,
step by step, then compares it with our own pipeline (`audio_engine/`, MFCC + MLflow).

**Professor's TP3 in 5 steps**
1. Split the data into **3 folds** with `KFold`, separately for `normal` and `abnormal`.
2. Features = **Mel spectrogram** in dB (course 2), flattened, first **1200 values**.
3. Model = **SVM with the rbf kernel** (`svm.SVC(kernel="rbf")`).
4. Metric = **macro precision** + confusion matrix.
5. Experiment: only **5, 10, 15, 20 files per class**, then plot the mean macro precision.

**What we change (and why)**
- No copying of `.wav` files into `train_kfold1/`, `test_kfold1/`... : we keep the **indices** of the files (same result, no duplicates on disk).
- We fix 3 bugs of the original notebook (explained in step 4).
- We add the metrics that matter in medicine (**recall** of the abnormal class, specificity, ROC-AUC).
- Every run is tracked in **MLflow** (experiment `TP3-Mel-Prof`).

> Run it with the `.venv` kernel (Python 3.11), from the project folder.

## 0. Imports and settings

In [ ]:
# Setup: run from the repository root, so paths like ourdataset/train and `import audio_engine` work
import os
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

In [ ]:
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import KFold
from sklearn.svm import SVC

# Reused from our project: the 6 metrics of the course and our SVM pipeline (scaler + balanced SVM)
from audio_engine.train_svm import compute_metrics, make_pipeline, metrics_figure, roc_figure

%matplotlib inline

DATA_DIR = Path("ourdataset/train")   # built by notebooks/01_eda_data_wrangling.ipynb
CROSS_NUMBER = 3                      # 3 folds, like the professor
RANDOM_STATE = 10                     # same seed as the professor
SELECTIONS = [5, 10, 15, 20]          # number of files per class used for training and for testing

# Mel spectrogram parameters of the professor (course 2)
N_FFT = 2048       # window size (samples)
HOP_LENGTH = 512   # step between two windows (samples)
N_MELS = 128       # number of Mel bands
N_VALUES = 1200    # length of the feature vector (first 1200 values of the flattened spectrogram)

LABELS = {"normal": 0, "abnormal": 1}  # positive class = 1 = abnormal

ImportError: cannot import name 'metrics_figure' from 'audio_engine.train_svm' (c:\Users\ASUS\Documents\3_éme_GMAM\Fm\Data-Mining\audio_engine\train_svm.py)

: 

## 1. The data

The two classes of `ourdataset/train` (built in TP1). The dataset is **imbalanced**: about 77 % normal / 23 % abnormal.

In [ ]:
normal = np.array(sorted(str(p) for p in (DATA_DIR / "normal").glob("*.wav")))
abnormal = np.array(sorted(str(p) for p in (DATA_DIR / "abnormal").glob("*.wav")))

total = len(normal) + len(abnormal)
print(f"normal:   {len(normal)} files ({len(normal) / total:.0%})")
print(f"abnormal: {len(abnormal)} files ({len(abnormal) / total:.0%})")

## 2. Split into 3 folds (`KFold`)

Like the professor, `KFold(n_splits=3, shuffle=True, random_state=10)` is applied **to each class separately**,
so every fold keeps the 77 / 23 ratio (it is a "manual" stratification).

The professor then took the *first* files of each folder. Here we **shuffle each list once with a fixed seed**
before taking the first `n` files: otherwise the first files in alphabetical order all come from the same source
(PASCAL), and the result would depend on the file names.

In [ ]:
kf = KFold(n_splits=CROSS_NUMBER, shuffle=True, random_state=RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

folds = []
for (n_train, n_test), (a_train, a_test) in zip(kf.split(normal), kf.split(abnormal)):
    folds.append({
        "train": {"normal": rng.permutation(normal[n_train]), "abnormal": rng.permutation(abnormal[a_train])},
        "test": {"normal": rng.permutation(normal[n_test]), "abnormal": rng.permutation(abnormal[a_test])},
    })

for k, fold in enumerate(folds, start=1):
    train_files = set(fold["train"]["normal"]) | set(fold["train"]["abnormal"])
    test_files = set(fold["test"]["normal"]) | set(fold["test"]["abnormal"])
    assert train_files.isdisjoint(test_files)   # a file is never in train AND test of the same fold
    print(f"fold {k}: train = {len(fold['train']['normal'])} normal + {len(fold['train']['abnormal'])} abnormal | "
          f"test = {len(fold['test']['normal'])} normal + {len(fold['test']['abnormal'])} abnormal")

## 3. Features: the Mel spectrogram (course 2)

For each file:
1. `librosa.load` reads the signal (resampled to 22 050 Hz by default, like the professor).
2. `melspectrogram`: Fourier transform on windows of 2048 samples every 512 samples, then 128 Mel filters
   → a matrix **(128 Mel bands × time frames)**.
3. `power_to_db`: convert the energy to decibels (log scale, closer to how we hear).
4. `flatten()[:1200]`: the professor turns the matrix into one long vector and keeps the first 1200 values.

> Bug fixed: the professor wrote `melspectrogram(signal, ...)`. Since librosa 0.10 the signal must be passed
> as `y=signal`, otherwise it crashes.

In [ ]:
def mel_spectrogram_db(path):
    signal, rate = librosa.load(path)  # default sr = 22050 Hz
    S = librosa.feature.melspectrogram(y=signal, sr=rate, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    return librosa.power_to_db(S, ref=np.max), rate


def mel_features(path):
    S_DB, _ = mel_spectrogram_db(path)
    return S_DB.flatten()[:N_VALUES]


# A cache, so a file used in several experiments is read only once
_cache = {}


def features_of(paths):
    for p in paths:
        if p not in _cache:
            _cache[p] = mel_features(p)
    return np.array([_cache[p] for p in paths])

### What do the "first 1200 values" contain?

`flatten()` reads the matrix **row by row**: first all the time frames of Mel band 0, then all those of band 1, etc.
So the first 1200 values are mostly the **lowest Mel band over time**, and *how much* time depends on the length
of the recording. Let's look at one example.

In [ ]:
example = normal[0]
S_DB, rate = mel_spectrogram_db(example)
n_bands, n_frames = S_DB.shape
print(f"{Path(example).name}: spectrogram shape = {S_DB.shape} (Mel bands x time frames)")
print(f"first {N_VALUES} values = {N_VALUES / n_frames:.1f} Mel band(s) out of {n_bands}")

fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(S_DB, sr=rate, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", ax=ax)
fig.colorbar(img, ax=ax, format="%+2.0f dB")
ax.set_title(f"Mel spectrogram (dB) - {Path(example).name}")
plt.show()

## 4. The professor's experiment: `SVC(kernel="rbf")` with 5, 10, 15, 20 files per class

For each number of files `n` and each fold `k`: train on `n` normal + `n` abnormal files of the train part,
test on `n` normal + `n` abnormal files of the test part.

**Bugs of the original notebook fixed here**
1. The features were saved in `train_kfoldK/mel_features_svm_a.data` but **reloaded from another folder**
   (`train/mel_features_svm_a.data`): the SVM could be trained on the wrong features. Here nothing goes through files.
2. `list_macro_precision` was never emptied: the "mean for 10 files" also contained the results for 5 files.
   Here every result is one row of a table, and we average per `n`.
3. `classification_report(..., target_names=["abnormal", "normal"])`: the labels are sorted (0, 1), so class 0
   (normal) was called "abnormal". Here the names follow the labels.

In [ ]:
METRICS = ["accuracy", "precision", "recall", "f1", "specificity", "roc_auc", "macro_precision"]


def select(fold, part, n):
    # n abnormal (label 1) + n normal (label 0) files of the "train" or "test" part of a fold
    paths = list(fold[part]["abnormal"][:n]) + list(fold[part]["normal"][:n])
    y = np.array([1] * n + [0] * n)
    return paths, y


def run_experiment(make_model, name, featurize=features_of):
    # featurize: function "list of paths -> feature matrix" (Mel features by default)
    rows = []
    for n in SELECTIONS:
        for k, fold in enumerate(folds, start=1):
            train_paths, y_train = select(fold, "train", n)
            test_paths, y_test = select(fold, "test", n)

            model = make_model()
            model.fit(featurize(train_paths), y_train)
            X_test = featurize(test_paths)
            y_pred = model.predict(X_test)
            scores = model.decision_function(X_test)  # distance to the hyperplane, for ROC-AUC

            metrics = compute_metrics(y_test, y_pred, scores)  # recall/precision of the abnormal class
            report = classification_report(y_test, y_pred, labels=[0, 1], target_names=["normal", "abnormal"],
                                           output_dict=True, zero_division=0)
            metrics["macro_precision"] = report["macro avg"]["precision"]  # the professor's metric
            rows.append({"model": name, "n_files": n, "fold": k, **metrics,
                         "cm": confusion_matrix(y_test, y_pred, labels=[0, 1]),
                         "y_test": y_test, "scores": scores})  # kept for the ROC curves
        print(f"{name}: n = {n} done")
    return pd.DataFrame(rows)


prof = run_experiment(lambda: SVC(kernel="rbf"), "prof")
prof[["n_files", "fold"] + METRICS].round(3)

### Mean over the 3 folds for each number of files

In [ ]:
prof_mean = prof.groupby("n_files")[METRICS].mean()
prof_mean.round(3)

### The professor's figure: mean macro precision vs number of files

We add the **recall of the abnormal class**: it tells how many sick hearts the model finds.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(SELECTIONS, prof_mean["macro_precision"], "o-", label="mean macro precision")
ax.plot(SELECTIONS, prof_mean["recall"], "s--", label="mean recall (abnormal)")
ax.set_xlabel("wavs selection number per class")
ax.set_ylabel("score")
ax.set_ylim(0, 1.05)
ax.set_xticks(SELECTIONS)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Professor's TP3: SVC(rbf) on Mel features")
plt.show()

### Confusion matrices with 20 files per class

In [ ]:
def plot_confusion_matrices(results, n, title, show=True):
    fig, axes = plt.subplots(1, CROSS_NUMBER, figsize=(12, 3.5))
    for ax, (_, row) in zip(axes, results[results["n_files"] == n].iterrows()):
        ConfusionMatrixDisplay(row["cm"], display_labels=["normal", "abnormal"]).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(f"fold {row['fold']}")
    fig.suptitle(title)
    fig.tight_layout()
    if show:
        plt.show()
    return fig  # returned so it can also be saved in MLflow


plot_confusion_matrices(prof, 20, "Professor's model (SVC rbf, no scaling) - 20 files per class")

**How to read it:** rows = true class, columns = prediction. The cell (abnormal, normal) is the number of
**false negatives**: sick hearts called "normal", the worst error in screening.

**Result:** about **0.5 everywhere**, the same as tossing a coin. With 10 to 40 training files and 1200 values
per file, and features that do not mean the same thing from one file to another (see step 3), the SVM cannot learn.

## 5. Same files, same folds, better features: MFCC (our `audio_engine`)

To find *where* the problem is, we change **only the features**: for each file we use the 30 features of
`audio_engine/features.py` (13 MFCC mean + std, ZCR and RMS mean/std, on 5 s segments at 2000 Hz),
averaged over the segments of the file. Same files, same folds, same rbf SVM.

The model is our pipeline `StandardScaler + SVC(rbf, class_weight="balanced")`: MFCC values have very different
scales, and an SVM works with distances, so scaling is needed. (`class_weight` changes nothing here, because
the selection already takes `n` normal + `n` abnormal files: the classes are balanced.)

In [ ]:
from audio_engine.features import load_audio, segment_features, split_segments

_mfcc_cache = {}


def mfcc_features_of(paths):
    for p in paths:
        if p not in _mfcc_cache:
            segments = split_segments(load_audio(p))                                  # 5 s segments at 2000 Hz
            _mfcc_cache[p] = np.mean([segment_features(s) for s in segments], axis=0)  # 30 values per file
    return np.array([_mfcc_cache[p] for p in paths])


ours = run_experiment(lambda: make_pipeline({"kernel": "rbf", "C": 1.0, "gamma": "scale"}), "mfcc",
                      featurize=mfcc_features_of)
ours_mean = ours.groupby("n_files")[METRICS].mean()

comparison = pd.DataFrame({
    "Mel (professor) recall": prof_mean["recall"],
    "MFCC (ours) recall": ours_mean["recall"],
    "Mel (professor) macro_precision": prof_mean["macro_precision"],
    "MFCC (ours) macro_precision": ours_mean["macro_precision"],
    "Mel (professor) roc_auc": prof_mean["roc_auc"],
    "MFCC (ours) roc_auc": ours_mean["roc_auc"],
})
comparison.round(3)

In [ ]:
def mel_vs_mfcc_figure():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, metric in zip(axes, ["macro_precision", "recall", "roc_auc"]):
        ax.plot(SELECTIONS, prof_mean[metric], "o-", label="Mel flatten[:1200] + SVC (professor)")
        ax.plot(SELECTIONS, ours_mean[metric], "s-", label="MFCC mean/std + scaler + SVC (ours)")
        ax.axhline(0.5, color="grey", linestyle=":", linewidth=1)   # chance level
        ax.set_title(f"mean {metric}")
        ax.set_xlabel("wavs selection number per class")
        ax.set_xticks(SELECTIONS)
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.3)
    axes[0].legend(loc="lower right")
    return fig


mel_vs_mfcc_figure()
plt.show()

plot_confusion_matrices(ours, 20, "MFCC features + scaler + SVC rbf - 20 files per class")

## 6. Track everything in MLflow

Experiment **`TP3-Mel-Prof`**: one parent run per (features, number of files) with the mean and std over the
3 folds, and 3 nested runs (one per fold) with the metrics.

The numbers are also saved **as graphics** (tab **Artifacts** of each run):
- fold run: `confusion_matrix.png`, `roc_curve.png`
- parent run: `metrics_per_fold.png` (bars = mean ± std, dots = folds), `confusion_matrices.png`,
  `roc_curves.png` and `mel_vs_mfcc.png` (the comparison chart of step 5)

Open the UI: `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001` → <http://localhost:5001>.
In the table, click the **+** next to a parent run to see its 3 folds.

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("TP3-Mel-Prof")

COMMON_PARAMS = {"cv": f"KFold({CROSS_NUMBER}) per class, random_state={RANDOM_STATE}", "kernel": "rbf"}
MODEL_PARAMS = {
    "prof": {"features": "mel-spectrogram dB, flatten()[:1200]", "n_fft": N_FFT, "hop_length": HOP_LENGTH,
             "n_mels": N_MELS, "model": "SVC(kernel='rbf')", "scaler": "none"},
    "mfcc": {"features": "13 MFCC mean+std, ZCR, RMS (30), mean over 5 s segments", "n_fft": 256,
             "hop_length": 64, "n_mels": 40, "model": "Pipeline(StandardScaler, SVC(rbf, balanced))",
             "scaler": "StandardScaler"},
}

def log_and_close(fig, path):
    mlflow.log_figure(fig, path)  # saved as a .png in the Artifacts tab of the current run
    plt.close(fig)


for results in [prof, ours]:
    for (name, n), group in results.groupby(["model", "n_files"]):
        with mlflow.start_run(run_name=f"{name}-n{n}"):
            mlflow.log_params({**COMMON_PARAMS, **MODEL_PARAMS[name], "n_files_per_class": n})
            for _, row in group.iterrows():
                with mlflow.start_run(run_name=f"{name}-n{n}-fold{row['fold']}", nested=True):
                    mlflow.log_params({**COMMON_PARAMS, **MODEL_PARAMS[name], "n_files_per_class": n,
                                       "fold": row["fold"]})
                    mlflow.log_metrics({m: row[m] for m in METRICS})
                    fig = ConfusionMatrixDisplay(row["cm"], display_labels=["normal", "abnormal"]).plot(cmap="Blues").figure_
                    log_and_close(fig, "confusion_matrix.png")
                    log_and_close(roc_figure([(row["y_test"], row["scores"])], f"ROC - {name} n={n} fold {row['fold']}"),
                                  "roc_curve.png")
            mlflow.log_metrics(group[METRICS].mean().to_dict())                                   # same names as the folds
            mlflow.log_metrics({f"{m}_std": v for m, v in group[METRICS].std(ddof=0).items()})   # stability over folds

            # The same numbers as graphics
            fold_metrics = [{m: row[m] for m in METRICS} for _, row in group.iterrows()]
            log_and_close(metrics_figure(fold_metrics, f"{name}, {n} files per class - metrics over 3 folds"),
                          "metrics_per_fold.png")
            log_and_close(plot_confusion_matrices(results, n, f"{name}, {n} files per class", show=False),
                          "confusion_matrices.png")
            log_and_close(roc_figure([(row["y_test"], row["scores"]) for _, row in group.iterrows()],
                                     f"ROC curves - {name}, {n} files per class"), "roc_curves.png")
            log_and_close(mel_vs_mfcc_figure(), "mel_vs_mfcc.png")

print("Logged:", len(prof) + len(ours), "fold runs in experiment TP3-Mel-Prof")

## 7. Comparison with our full pipeline (MFCC, all the data)

Our `audio_engine` (run with `uv run python -m audio_engine.train_svm`) uses **all 3784 files** (15 781 segments),
grouped folds and a balanced SVM. Its results are read back from MLflow (experiment `TP3-SVM`, last `svm-rbf` run).

In [ ]:
last_rbf = mlflow.search_runs(experiment_names=["TP3-SVM"], filter_string="tags.mlflow.runName = 'svm-rbf'",
                              order_by=["start_time DESC"], max_results=1)

SHOWN = ["accuracy", "precision", "recall", "f1", "specificity", "roc_auc"]
table = pd.DataFrame({
    "1. professor: Mel, 20 files/class": prof_mean.loc[20, SHOWN],
    "2. MFCC, 20 files/class": ours_mean.loc[20, SHOWN],
})
if len(last_rbf):
    table["3. MFCC, all 3784 files"] = [last_rbf[f"metrics.{m}_mean"].iloc[0] for m in SHOWN]
else:
    print("No TP3-SVM run found: run `uv run python -m audio_engine.train_svm` first.")
table.round(3)

## 8. Conclusion

Results with 20 files per class (mean over the 3 folds, seed 10):

| | accuracy | recall (abnormal) | specificity | ROC-AUC |
|---|---|---|---|---|
| Mel `flatten()[:1200]` + SVC (professor) | 0.49 | 0.65 | 0.33 | 0.49 |
| MFCC mean/std + scaler + SVC (ours) | 0.79 | 0.88 | 0.70 | 0.86 |
| MFCC, all 3784 files, 5 s segments (`audio_engine`) | 0.88 | 0.90 | 0.87 | 0.95 |

1. **The professor's setup stays at chance level (≈ 0.5)** for every number of files: the SVM learns nothing.
   With so few test files the scores also jump a lot between folds (with 10 test files, one file = 10 %).
2. **The features are the problem, not the SVM.** With the *same* files, folds and kernel, only changing the
   features takes the ROC-AUC from 0.49 to 0.86. `flatten()[:1200]` keeps only the first ~3 Mel bands (lowest
   frequencies), and position *i* is not the same moment in two recordings. MFCC mean/std is a fixed-size
   summary that means the same thing for every file.
3. **More data helps too**: with all the files, our pipeline reaches recall ≈ 0.90 and ROC-AUC ≈ 0.95.
4. `class_weight="balanced"` only matters on the real, imbalanced data (77 / 23), not on the balanced
   n + n selection of this TP.
5. In medicine we choose the model by the **recall of the abnormal class**: a false negative sends a sick
   patient home.

**Bugs of the original notebook (fixed here):** wrong folder when reloading the features, `list_macro_precision`
never emptied between selections, swapped class names in `classification_report`, `melspectrogram(signal)`
without `y=` (crash with recent librosa), plus all the Google Colab / Drive code (everything runs locally here).

**Oral questions to be ready for**
- Why `KFold` on each class separately? → to keep the 77 / 23 ratio in each fold (manual stratification).
- Why is macro precision not enough? → it averages both classes; we must look at the recall of the abnormal
  class, because missing a sick heart is the worst error.
- What is the kernel trick? → the rbf kernel computes similarities as if the data were mapped to a very
  high-dimensional space, where a hyperplane can separate them, without ever computing that mapping.